# Do rule-based title features improve YouTube view predictions?

We predict `log_views` using Ridge regression. Our question is whether 14 title
features, such as title length, uppercase ratio, and exclamation mark count,
improve predictions beyond video age, channel baseline, category, and publication week.

We evaluate two settings: later videos from known channels and videos from unseen
channels. Within each setting, both models use the same training and test videos.

In [12]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ytclickbait.config import load_config
from ytclickbait.rule_based_model_inputs import (
    merge_rule_inputs,
    score_rule_predictions,
)

## 1. Build the modeling dataset

Each row represents one video. We join video metadata and `log_views` from
`videos.parquet`, split assignments and channel baselines from `split.parquet`,
and 14 title features from `rule_features.parquet`.

The join uses `video_id`. Before merging, `merge_rule_inputs()` checks that the
three tables contain the same video IDs and that each ID appears once per table.

In [13]:
cfg = load_config()

videos = pd.read_parquet(cfg.file("videos"))
split = pd.read_parquet(cfg.file("split"))
rules = pd.read_parquet(cfg.file("rule_features"))

# Use the same merge function covered by the input validation tests.
df = merge_rule_inputs(videos, split, rules)
rule_cols = rules.columns.drop("video_id").tolist()

print(f"Modeling dataset: {len(df):,} videos, {len(rule_cols)} rule features")
display(df[["title", "log_views", "split_time", "n_exclaim", "caps_ratio"]].head())

Modeling dataset: 78,180 videos, 14 rule features


,title,log_views,split_time,n_exclaim,caps_ratio
0,Jadon Sancho • Magical Skills & Goals,12.979280,train,0,0.172414
1,Migos - Avalanche (Official Video),15.735846,excluded,0,0.148148
2,MY TESLA PAYS FOR ITSELF,13.245354,excluded,0,1.000000
3,Supporting Actress in a Comedy: 73rd Emmys,13.433679,train,0,0.121212
4,San Francisco 49ers vs. Arizona Cardinals Game...,14.478602,test,0,0.185185


## 2. Define the model comparison

For each split, we fit two Ridge models. The baseline uses four controls.
The second model adds the 14 title features. We keep the training rows, test
rows, preprocessing procedure, and Ridge settings the same within each split.

We report test RMSE on the log-view scale and test R². Lower RMSE and higher
R² indicate better predictions on the same test videos.

In [14]:
def compare_rule_models(train, test, channel_baseline_col, rule_cols):
    """Compare two Ridge models on the same training and test videos."""
    numeric_controls = ["log_age_days", channel_baseline_col]
    category_cols = ["category_name", "publish_week"]
    rows = []

    for model_name, extra_cols in [
        ("Controls only", []),
        ("Controls + 14 rule features", rule_cols),
    ]:
        numeric_cols = numeric_controls + extra_cols
        input_cols = numeric_cols + category_cols

        preprocess = ColumnTransformer([
            ("numeric", StandardScaler(), numeric_cols),
            ("category", OneHotEncoder(handle_unknown="ignore"), category_cols),
        ])

        # Fit preprocessing and Ridge using training videos only.
        model = make_pipeline(preprocess, Ridge(alpha=1.0, solver="lsqr"))
        model.fit(train[input_cols], train["log_views"])

        # Score predictions against the actual log views of the same test videos.
        predictions = model.predict(test[input_cols])
        scores = score_rule_predictions(test["log_views"], predictions)

        rows.append({
            "Model": model_name,
            "RMSE": scores["RMSE"],
            "R²": scores["R²"],
        })

    return pd.DataFrame(rows).set_index("Model")

## 3. Predict later videos from known channels

Using `split_time`, we train on earlier videos and evaluate on later videos from
known channels. Only rows labeled `train` or `test` enter this experiment.

In [15]:
time_train = df.loc[df["split_time"].eq("train")]
time_test = df.loc[df["split_time"].eq("test")]

print(f"Training videos: {len(time_train):,}")
print(f"Test videos: {len(time_test):,}")
print(f"Excluded videos: {df['split_time'].eq('excluded').sum():,}")

time_results = compare_rule_models(
    train=time_train,
    test=time_test,
    channel_baseline_col="channel_baseline_split_time",
    rule_cols=rule_cols,
)

display(time_results.round(4))

Training videos: 42,204
Test videos: 11,487
Excluded videos: 24,489


,RMSE,R²
Model,,
Controls only,0.7385,0.5953
Controls + 14 rule features,0.7350,0.5992


## 4. Predict videos from unseen channels

Using `split_group`, we evaluate videos from channels that never appear in
training. We check for channel overlap before fitting either model.

In [16]:
group_train = df.loc[df["split_group"].eq("train")]
group_test = df.loc[df["split_group"].eq("test")]

# Both sets must contain videos, and their channel IDs must be disjoint.
assert not group_train.empty and not group_test.empty
assert set(group_train["channel_id"]).isdisjoint(group_test["channel_id"])

print(f"Unseen test channels: {group_test['channel_id'].nunique():,}")
print(f"Test videos: {len(group_test):,}")

group_results = compare_rule_models(
    train=group_train,
    test=group_test,
    channel_baseline_col="channel_baseline_split_group",
    rule_cols=rule_cols,
)

display(group_results.round(4))

Unseen test channels: 2,387
Test videos: 14,743


,RMSE,R²
Model,,
Controls only,1.2048,0.1180
Controls + 14 rule features,1.2029,0.1207


## 5. Compare the contribution of title features

For each split, we subtract the title-feature model's RMSE from the baseline
RMSE. A positive value means that adding title features reduced prediction
error. We also report the increase in R² in percentage points.

In [17]:
comparison_rows = []

for split_name, results in [
    ("Later videos from known channels", time_results),
    ("Videos from unseen channels", group_results),
]:
    baseline = results.loc["Controls only"]
    with_rules = results.loc["Controls + 14 rule features"]

    comparison_rows.append({
        "Evaluation setting": split_name,
        "Controls RMSE": baseline["RMSE"],
        "Controls + rules RMSE": with_rules["RMSE"],
        "RMSE reduction": baseline["RMSE"] - with_rules["RMSE"],
        "R² increase (pp)": 100 * (with_rules["R²"] - baseline["R²"]),
    })

comparison = pd.DataFrame(comparison_rows).set_index("Evaluation setting")
display(comparison.round(4))

,Controls RMSE,Controls + rules RMSE,RMSE reduction,R² increase (pp)
Evaluation setting,,,,
Later videos from known channels,0.7385,0.7350,0.0035,0.3857
Videos from unseen channels,1.2048,1.2029,0.0019,0.2727


## Conclusion

Adding the 14 title features reduced test RMSE by 0.0035 for later videos from
known channels and by 0.0019 for videos from unseen channels. The observed
improvement was small in both settings and larger for known channels.

These results describe the joint predictive contribution of the 14 features
in this dataset. Estimating the effect of editing an individual title would
require a different study design.